## Init

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, lit

In [0]:
import sys
import os

current_directory = os.getcwd()
if current_directory not in sys.path:
    sys.path.append(current_directory)
    
from script.utils.config import cities_config

## Create DataFrame from static dictionary

In [0]:
df = spark.createDataFrame(cities_config)

windowSpec = Window.orderBy(lit('state'))
df_state_code = (
    df
    .withColumn(
        "state_code", 
        row_number().over(windowSpec)
    )
)

df_cities = (
    df_state_code
    .select(
        "state_code", "state", "city", "lat", "lon"
    )
)

## Write in silver table 

In [0]:
(
    df_cities.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("weather.silver_cities")
)

## Sanity check

In [0]:
%sql
select * 
from weather.silver_cities